# 47. 부분조건 활용과 전체 복귀 비교

46번의 선정 경계를 고정하고, 구조화 AND 전체를 찾지 못했을 때 A 전체 비구조화 복귀와 B 부분일치 우선+원래 허용 범위 보완을 비교합니다.

기존48개 개발 질문과 원문을 확인한4개 통제 질문을 사용합니다. DB 누락·불명확은 별도 메모리 fact view로 모의하며 원래 DB와 OCR은 수정하지 않습니다. 서로 다른 카드의 혜택을 합쳐 전체 충족이라고 판정하지 않습니다.

두 청킹은 운영에서 만든31번 청크를 그대로 사용합니다. 검색 엔진을 열 때의 쓰기에서 과거 인덱스를 보존하려고 이 실험의 private copy를 씁니다.

새 공용 보조 코드의 이유:47~49에서 같은 운영 검색·재정렬·임베딩 캐시·답변 연결을 반복 사용합니다. 기존34번의 순수 budget5 함수와 운영 검색 클래스를 재사용하며, 경로 이동 전 계약을 우회하거나 과거 실행기를 재실행하지 않습니다. 실행·봉인·평가·HTML 해석은 노트북이 담당합니다.

In [1]:
from pathlib import Path
import importlib.util,sys,json,hashlib,time,os,copy
ROOT=Path.cwd()
if not (ROOT/'notebooks').exists():ROOT=ROOT.parent
sys.dont_write_bytecode=True
def load(name,filename):
    spec=importlib.util.spec_from_file_location(name,ROOT/'notebooks/scripts'/filename)
    module=importlib.util.module_from_spec(spec);sys.modules[name]=module;spec.loader.exec_module(module)
    return module
P=load('stage47_common','46_preproduction_common.py')
PATH=P.F.STAGES[47];PATH.mkdir(exist_ok=True);P.F.initialize(47)
assert P.F.read_json(P.F.STAGES[46]/'completion_status.json')['mandatory_regression_pass']
selected46=P.F.read_json(P.F.STAGES[46]/'selected_policy.json')
assert selected46['hard_target_literal_check'] is False
print('46번 필수 게이트 통과 및 선정 설정 확인')


46번 필수 게이트 통과 및 선정 설정 확인


In [2]:
R=load('stage47_runtime','47_retrieval_runtime.py')
facts=[json.loads(line) for line in (P.DATA/'30_structured_benefit_db/facts.jsonl').read_text().splitlines()]
prior=P.DATA/'42_expanded_query_evaluation'
catalog=P.F.read_json(prior/'catalog.json')['catalog']
public=P.F.read_json(prior/'questions.json')['queries']
extra=[
 dict(query_id='P01',question='커피 전문점 10% 이상 할인과 넷플릭스 20% 이상 할인을 한 카드에서 모두 받고 싶어',view='missing_netflix'),
 dict(query_id='P02',question='커피 10% 이상 할인, 넷플릭스 20% 이상 할인이 동시에 가능한 카드가 있어?',view='missing_both_gold'),
 dict(query_id='P03',question='신한카드 Discount Plan+에서 커피 10% 이상 할인과 넷플릭스 20% 이상 할인이 둘 다 되는지 조건까지 설명해줘',view='unknown_netflix_rate'),
 dict(query_id='P04',question='커피 10% 이상 할인과 넷플릭스 90% 이상 할인이 모두 되는 카드 한 장',view='unchanged_negative'),
]
queries=[dict(**q,view='unchanged') for q in public]+extra
profiles=['card_page_section_benefit','parent_child_bundle'];modes=['A','B']
annotations=P.F.read_json(prior/'dataset.json')['queries']
# 두 원문 fact는 공통조건을 포함한 혜택 관계를 확인한 출처입니다.
witness_ids=['fact_6faa53714a38539c06c8677a','fact_7923e642f3e8c43f426fade3']
witnesses=[r for r in facts if r['fact_id'] in witness_ids]
assert len(witnesses)==2
source_card='shinhan/Shinhan_Discount_Plan+_20250509'
extra_annotations=[dict(query_id=q['query_id'],gold_cards=[source_card] if q['query_id']!='P04' else [],
  negative_control=q['query_id']=='P04',expected_behavior='두 혜택을 같은 카드에서 확인; 원문 조건 유지' if q['query_id']!='P04' else '넷플릭스90%를 충족한다고 추천하지 않음',
  source_fact_ids=witness_ids,split='controlled_development',
  source_notes='커피 DAY10%는07~15시·오프라인·임대매장제외·각영역일1회. 구독20%는공식홈페이지정기결제·인앱/통신합산제외; 1회1만원은할인액한도로단정하지 않음') for q in extra]
# 평가용 정답 계획은 원문 기준 리터럴이며 LLM 요청에 넣지 않습니다.
for annotation in extra_annotations:
    target_rate=90 if annotation['query_id']=='P04' else 20
    def eval_requirement(target,rate):
        return dict(targets=[target],action='discount',min_percent=rate,max_spend_krw=None,
                    min_cap_krw=None,cap_period='any',exclude_targets=[])
    annotation.update(task='detail' if annotation['query_id']=='P03' else 'recommend',
      subgroup='partial_control',calculations=[],answer_checks=[],
      gold_plan=dict(route='structured_then_unstructured',groups=[[eval_requirement('커피',10),eval_requirement('넷플릭스',target_rate)]],
        card_keys=[source_card] if annotation['query_id']=='P03' else [],unresolved=[]))
PROMPT=load('stage47_prompt','33_query_routing.py').INSTRUCTIONS+'\n카드 catalog:\n'+P.F.canonical(catalog)
source_paths=[ROOT/'notebooks/scripts'/name for name in ['47_retrieval_runtime.py','46_preproduction_common.py',
 '34_bundle_experiment.py','34_followup_common.py','35_answer_experiment.py','36_cost_accounting.py','33_query_routing.py',
 '32_search_experiment.py','30_build_structured_db.py','39_answer_rubric.py']]
source_paths += [ROOT/name for name in ['packages/rag-core/src/pickcardu_rag/retrieval.py',
 'packages/rag-core/src/pickcardu_rag/answering.py','services/rag-api/src/pickcardu_rag_api/index.py']]
manifest=dict(experiment=47,status='authorized',questions_sha256=P.sha(queries),evaluation_annotations_sha256=P.sha(annotations+extra_annotations),
 questions=[q['query_id'] for q in queries],question_conditions=52,profiles=profiles,modes=modes,logical_answer_conditions=208,
 historical_execution_sha256=P.F.read_json(prior/'execution_protocol.json')['sha256'],
 new_parser_conditions=4,maximum_parser_requests=4,maximum_embedding_requests=1,maximum_answer_requests=416,
 maximum_physical_requests=421,maximum_validation_retries_per_answer=1,maximum_network_retries=0,
 maximum_logical_retrieval_calls=208,maximum_backend_component_calls=832,maximum_rerank_pairs=4160,
 global_cost_cap_usd='8',stage_cost_cap_usd='1.50',
 model='gpt-6-luna',reasoning='medium',max_output_tokens=3000,answer_output=2400,parser_output=3000,
 allowed_identities={'parser':[q['query_id']+':parser' for q in extra],
   'answer':[f"{q['query_id']}:{profile}:{mode}" for q in queries for profile in profiles for mode in modes]},
 allowed_request_by_identity={q['query_id']+':parser':P.sha([dict(role='user',content=q['question'])]) for q in extra},
 allowed_embedding_text_sha256=[P.sha(q['question']) for q in queries],
 parser_instructions_sha256=P.sha(PROMPT),parser_schema_sha256=P.sha(P.E.QueryPlan.model_json_schema()),
 source_sha256={str(f.relative_to(ROOT)):hashlib.sha256(f.read_bytes()).hexdigest() for f in source_paths},
 data_sha256={str(f.relative_to(ROOT)):hashlib.sha256(f.read_bytes()).hexdigest() for f in [
 P.DATA/'30_structured_benefit_db/facts.jsonl',P.F.STAGES[46]/'selected_policy.json',prior/'questions.json',
 *[P.DATA/'31_operational_chunking_comparison'/f'{profile}_chunks.jsonl' for profile in profiles],
 *[P.DATA/'31_operational_chunking_comparison'/profile/filename for profile in profiles for filename in ['embeddings.npy','corpus.sqlite']],
 prior/'dataset.json',prior/'parser_results.json',prior/'combined_results.json',prior/'execution_protocol.json',prior/'protected_files.json',
 *[prior/folder/f"{q['query_id']}_{profile}_S_THEN_U.json" for folder in ['retrieval','answers'] for q in public for profile in profiles],
 *[source for profile in profiles for source in (P.DATA/'31_operational_chunking_comparison'/profile).rglob('*') if source.is_file()]]},
 fixed_parameters=dict(component_depth=50,candidate_depth=20,top_k=3,evidence_per_card=5,payload_bytes=64000,
  vector_weight=.4,parent_bundle='budget5',merge='modality-wise equal RRF k60 then unchanged operational hybrid/reranker'),
 fallback_A='original question within original trusted scope',
 recovery_B='only when no eligible full AND: union of true requirement cards prioritized plus original-scope search; no full-AND approval',
 unchanged_reuse='saved42 retrieval/answer only if current scope equals original scope; B exact same payload reuses A. Historical reuse is not fresh regeneration.',
 answer_payload_contract_required=True,operating_changes=False,new_document_embeddings=0,
 stop_rule='source/input/unknown cost/request cap/memory violations stop; fresh important B error rejects B',
 selection_gate='no new correct-card loss, full-AND/scope/important-condition/citation error; at least one source-supported improvement; otherwise A')
if (PATH/'execution_manifest.json').exists():assert P.F.read_json(PATH/'execution_manifest.json')==manifest
else:P.F.write_json(PATH/'execution_manifest.json',manifest)
P.F.write_json(PATH/'questions.json',dict(queries=queries,split='development'))
P.F.write_json(PATH/'evaluation_annotations.json',dict(queries=annotations+extra_annotations,source_witnesses=witnesses,never_sent_to_provider=True))
print('봉인:52질문조건×2프로필×2설정=208논리답변; 새 parser4·임베딩최대1·전체물리상한421')


봉인:52질문조건×2프로필×2설정=208논리답변; 새 parser4·임베딩최대1·전체물리상한421


In [3]:
# 검색 함수에서 만든 정답을 쓰지 않는 리터럴 회귀입니다.
fixture=P.F.read_json(P.DATA/'43_offline_validation/routing_fixtures.json')
def requirement(target,rate=10):
 return dict(targets=[target],action='discount',min_percent=rate,max_spend_krw=None,
  min_cap_krw=None,cap_period='any',exclude_targets=[])
value=dict(route='structured_then_unstructured',groups=[[requirement('커피'),requirement('넷플릭스')]],card_keys=[],unresolved=[])
missing=[r for r in fixture['facts'] if r['fields']['target']!='넷플릭스']
routing=R.selected_route('',value,missing,fixture['catalog'])
assert routing['reason']=='no_eligible_cards'
assert R.partial_candidates(value,missing,routing)==['A','B','I']
assert R.partial_candidates(value,fixture['facts'],R.selected_route('',value,fixture['facts'],fixture['catalog']))==[]
scoped=R.selected_route('',{**value,'card_keys':['A']},missing,fixture['catalog'],trusted_scope=['A'])
assert R.partial_candidates({**value,'card_keys':['A']},missing,scoped)==['A']
failed=R.selected_route('',value,missing,fixture['catalog'],parser_failed=True)
assert R.partial_candidates(value,missing,failed)==[]
from pickcardu_rag.retrieval import Candidate
merged=R.merge_components([Candidate('a',1,1),Candidate('b',.5,2)],[Candidate('b',1,1),Candidate('c',.5,2)],3)
assert [r.chunk_id for r in merged]==['b','a','c']
P.F.write_json(PATH/'offline_regression.json',dict(passed=True,cases=5,new_api_requests=0,
 purpose='partial priority and explicit scope cannot become full-AND approval; equalRRF literal ordering'))
print('부분조건/범위/실패/RRF 회귀5건 통과')

# 질문·반복 입력 결합과 사용량 미상 중단을 실제 API 경계에서 검사합니다.
api=R.BoundCache(47,'parser','P01:parser',allow_network=True)
kwargs=dict(instructions=PROMPT,input=[dict(role='user',content=extra[0]['question'])],
 text_format=P.E.QueryPlan,tools=[],store=False)
try:api.reserve(api.normalized({**kwargs,'input':[dict(role='user',content=extra[-1]['question'])]}))
except RuntimeError as exc:assert 'Identity and frozen provider payload differ' in str(exc)
else:raise AssertionError('다른 질문의 identity 사용이 허용됐습니다.')
original_ledger=R.P.ledger
try:
 R.P.ledger=lambda:dict(unknown_cost_records=1,known_cost_usd='0')
 try:api.reserve(api.normalized(kwargs))
 except RuntimeError as exc:assert 'Unknown prior request cost' in str(exc)
 else:raise AssertionError('비용 미상에서 유료 호출이 허용됐습니다.')
finally:R.P.ledger=original_ledger
print('API 입력 결합·비용 경계 검사2건 통과, 실제 새 요청0')


부분조건/범위/실패/RRF 회귀5건 통과
API 입력 결합·비용 경계 검사2건 통과, 실제 새 요청0


In [4]:
protection_path=PATH/'protected_files.json'
if not protection_path.exists():
    import subprocess
    tracked=subprocess.check_output(['git','ls-files','-z'],cwd=ROOT).decode().split('\0')
    protected={ROOT/name for name in tracked if name and (ROOT/name).is_file()}
    for base in ['docs','notebooks','data','services','packages','jobs']:
        for source in (ROOT/base).rglob('*'):
            if not source.is_file() or source.is_symlink():continue
            relative=str(source.relative_to(ROOT))
            if any(part in source.parts for part in ['__pycache__','node_modules','.next','execution_cache']):continue
            if relative.startswith(('notebooks/data/47_','notebooks/data/48_','notebooks/data/49_',
                'notebooks/data/00_notebook_layout_migration/','notebooks/47_','notebooks/48_','notebooks/49_',
                'notebooks/scripts/47_','notebooks/scripts/48_','notebooks/scripts/49_',
                'docs/report/47_','docs/report/48_','docs/report/49_')):continue
            if relative=='notebooks/notes/46_49_preproduction_test_plan.md':continue
            protected.add(source)
    protected={p for p in protected if str(p.relative_to(ROOT))!='notebooks/notes/46_49_preproduction_test_plan.md'}
    write_value=dict(files=[P.F.file_record(source) for source in sorted(protected)],
        frozen_after46_completion=True,excluded_execution_caches=True)
    P.F.write_json(protection_path,write_value)
print('47번 보존 기준 파일',len(P.F.read_json(protection_path)['files']))


47번 보존 기준 파일 25457


In [5]:
initial_api_record_count=len(list((PATH/'api_cache').glob('*.json')))
from types import SimpleNamespace
from pickcardu_rag.answering import serialize_answer_payload, _generated_answer_model
# 과거96개 답변은 원래 파일의 retrieval hash, provider input/schema/model/output와 대조합니다.
history=[]
for q in public:
    plan=P.F.read_json(prior/'parser_results.json')['records']
    value=next(r['plan'] for r in plan if r['query_id']==q['query_id'])
    scope=R.selected_route(q['question'],value,facts,catalog,trusted_scope=P.explicit_scope(q['question'],catalog))['scope']
    for profile in profiles:
        old_r,old_a,old_payload=R.verified_historical(q,profile,scope,manifest=manifest)
        history.append(dict(query_id=q['query_id'],profile=profile,passed=True,retrieval_sha256=P.sha(old_r),
            answer_sha256=P.sha(old_a),payload_sha256=old_payload))
assert len(history)==96
P.F.write_json(PATH/'historical_binding_verification.json',dict(records=history,new_api_requests=0))
# 실제 저장 provider 응답을 별도 시험 경로에 복사해 새 재개 검사를 시험합니다.
q=public[0];profile=profiles[0]
plan=next(r['plan'] for r in P.F.read_json(prior/'parser_results.json')['records'] if r['query_id']==q['query_id'])
scope=R.selected_route(q['question'],plan,facts,catalog)['scope']
old_r,old_a,payload=R.verified_historical(q,profile,scope,manifest=manifest)
identity=f"{q['query_id']}:{profile}:A";evidence=old_r['result']['evidence']
schema=_generated_answer_model(len(evidence));last=old_a['events'][-1]['key']
cache_source=next(P.DATA/name/'api_cache'/(last+'.json') for name in ['42_expanded_query_evaluation','30_structured_benefit_db']
 if (P.DATA/name/'api_cache'/(last+'.json')).exists())
record=P.F.read_json(cache_source);kwargs={k:v for k,v in record['request'].items() if k!='schema'};kwargs['text_format']=schema
cache=R.BoundCache(47,'answer',identity,allow_network=False)
testdir=PATH/'contract_tests/provider_fixture';testdir.mkdir(parents=True,exist_ok=True)
key=cache.key(kwargs)
P.F.write_json(testdir/(key+'.json'),record)
contract=dict(execution_manifest_sha256=P.sha(manifest),input_sha256_by_identity={identity:payload},
 schema_sha256_by_identity={identity:P.sha(schema.model_json_schema())})
row=dict(identity=identity,manifest_sha256=P.sha(manifest),payload_contract_sha256=P.sha(contract),
 input_sha256=payload,status='ok',answer=old_a['answer'],events=[dict(key=key)],request_keys=[key])
original_cache_class=R.BoundCache
def test_cache(*args,**kwargs):
    instance=original_cache_class(*args,**kwargs);instance.cache=testdir
    return instance
R.BoundCache=test_cache
try:
    R.validate_generated_answer(row,q['question'],evidence,stage=47,identity=identity,manifest=manifest,contract=contract)
    broken=copy.deepcopy(row);broken['answer']['answer_text']='저장 답변이 바뀜'
    try:R.validate_generated_answer(broken,q['question'],evidence,stage=47,identity=identity,manifest=manifest,contract=contract)
    except RuntimeError as exc:assert 'differs from restored provider output' in str(exc)
    else:raise AssertionError('변조 답변이 허용됐습니다.')
    corrupted=copy.deepcopy(record);corrupted['request']['input']=[dict(role='user',content='다른 질문')]
    P.F.write_json(testdir/(key+'.json'),corrupted)
    try:R.validate_generated_answer(row,q['question'],evidence,stage=47,identity=identity,manifest=manifest,contract=contract)
    except RuntimeError as exc:assert 'provider request/schema/model/usage mismatch' in str(exc)
    else:raise AssertionError('원응답 입력 변경이 허용됐습니다.')
finally:
    R.BoundCache=original_cache_class
    P.F.write_json(testdir/(key+'.json'),record)
# 복사본 바이트 변조 거절: 실제 index 대신 별도 test copy를 사용합니다.
source=P.DATA/'31_operational_chunking_comparison'/profiles[0]/'corpus.sqlite'
target=PATH/'contract_tests/copy_guard';target.mkdir(parents=True,exist_ok=True)
copy_path=target/'corpus.sqlite'
import shutil
shutil.copy2(source,copy_path)
expected=P.sha(source.read_bytes())
R.P.F.ensure_private_copy(source,copy_path,target,expected)
with copy_path.open('ab') as stream:stream.write(b'corrupted-test-only')
try:R.P.F.ensure_private_copy(source,copy_path,target,expected)
except RuntimeError as exc:assert 'SHA mismatch' in str(exc)
else:raise AssertionError('복사본 변경이 허용됐습니다.')
assert len(list((PATH/'api_cache').glob('*.json')))==initial_api_record_count
P.F.write_json(PATH/'provenance_regression.json',dict(historical_valid=96,cache_answer_mutation_rejected=True,
 cache_input_mutation_rejected=True,private_copy_mutation_rejected=True,new_paid_api_records=0))
print('과거96개 원입력·출력 대조, 답변/입력/복사본 변조 거절 통과. 새 API0회')


과거96개 원입력·출력 대조, 답변/입력/복사본 변조 거절 통과. 새 API0회


In [6]:
# 유료 실행 전 실제 private index와 운영 검색의 소규모 동등성을 검사합니다.
index_runtime=R.Runtime(47)
parity=[]
for qid in ['E01','E14','E17']:
    q=next(q for q in public if q['query_id']==qid)
    value=next(r['plan'] for r in P.F.read_json(prior/'parser_results.json')['records'] if r['query_id']==qid)
    scope=R.selected_route(q['question'],value,facts,catalog,trusted_scope=P.explicit_scope(q['question'],catalog))['scope']
    vector=index_runtime.embeddings([q['question']],allow_network=False)[0]
    for profile in profiles:
        old_r,_,_=R.verified_historical(q,profile,scope,manifest=manifest)
        component_identity=P.sha(dict(question=q['question'],vector=P.sha(index_runtime.np.asarray(vector,dtype=index_runtime.np.float32).tobytes()),profile=profile,scope=scope,depth=50))
        seed=dict(experiment_component_identity=component_identity,
            experiment_paths={'original':{k:old_r['result']['trace'][k] for k in ['bm25','vector']}})
        result=index_runtime.search(q['question'],vector,profile,scope,baseline_result=seed)
        same_cards=result['cards']==old_r['result']['cards'];same_evidence=result['evidence']==old_r['result']['evidence']
        parity.append(dict(query_id=qid,profile=profile,cards_equal=same_cards,evidence_equal=same_evidence))
        assert same_cards and same_evidence,parity[-1]
        print('운영 검색 동등성',qid,profile,'통과',flush=True)
# 논리 index 계약을 손상시킨 재개가 거절되는지도 검사하고 원래 계약을 복원합니다.
profile=profiles[0];logical_path=PATH/'index_copies'/(profile+'_logical_identity.json')
saved=P.F.read_json(logical_path);broken={**saved,'chroma_metadata_sha256':'0'*64}
P.F.write_json(logical_path,broken);index_runtime.data.pop(profile)
try:
    try:index_runtime.profile(profile)
    except RuntimeError as exc:assert 'Resumed logical index identity differs' in str(exc)
    else:raise AssertionError('논리 index 계약 변조가 허용됐습니다.')
finally:P.F.write_json(logical_path,saved)
index_runtime.profile(profile)
P.F.write_json(PATH/'operating_parity.json',dict(records=parity,all_passed=all(r['cards_equal'] and r['evidence_equal'] for r in parity),
 logical_identity_corruption_rejected=True,new_api_requests=0,resources=index_runtime.resources(),
 note='원래 저장 component를 고정해 운영 후처리·budget5·소유권 결과를 비교. 새 ANN 검색 동일성 주장은 아님'))
print('private index 재개 변조 거절 및 복원 통과; 새 API0회')


/home/sms/miniconda3/envs/skn25/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


운영 검색 동등성 E01 card_page_section_benefit 통과


운영 검색 동등성 E01 parent_child_bundle 통과


운영 검색 동등성 E14 card_page_section_benefit 통과


운영 검색 동등성 E14 parent_child_bundle 통과


운영 검색 동등성 E17 card_page_section_benefit 통과


운영 검색 동등성 E17 parent_child_bundle 통과


private index 재개 변조 거절 및 복원 통과; 새 API0회


In [7]:
def write(name,value):P.F.write_json(PATH/name,value)
def frozen(path,value):
    if path.exists():assert P.F.read_json(path)==value, '기존 실행 입력이 바뀌었습니다: '+str(path)
    else:P.F.write_json(path,value)
def fact_view(kind):
    output=copy.deepcopy(facts)
    is_netflix=lambda r:'넷플릭스' in r['derived']['target_normalized']
    if kind in ['missing_netflix','missing_both_gold']:
        output=[r for r in output if not is_netflix(r)]
    if kind=='missing_both_gold':
        output=[r for r in output if r['fact_id']!='fact_6faa53714a38539c06c8677a']
    if kind=='unknown_netflix_rate':
        for r in output:
            if is_netflix(r):r['derived'].update(percent=None,percent_state='unknown')
    return output
# 모의 누락·불명확의 원문은 그대로 두고 실험 DB view만 바꿉니다.
view_audit={kind:dict(source_fact_count=len(facts),view_fact_count=len(fact_view(kind)),
 view_sha256=P.sha(fact_view(kind))) for kind in sorted({q['view'] for q in queries})}
frozen(PATH/'fact_view_contract.json',view_audit)
ALLOW_NETWORK=os.getenv('PREPROD47_ALLOW_NETWORK')=='1'
plans={r['query_id']:r for r in P.F.read_json(prior/'parser_results.json')['records']}
for q in extra:
    identity=q['query_id']+':parser'
    api=R.BoundCache(47,'parser',identity,allow_network=ALLOW_NETWORK)
    kwargs=dict(model='gpt-6-luna',instructions=PROMPT,input=[dict(role='user',content=q['question'])],
                text_format=P.E.QueryPlan,tools=[],store=False,max_output_tokens=3000)
    key=api.key(kwargs);destination=PATH/'plans'/(q['query_id']+'.json')
    if destination.exists():row=P.F.read_json(destination)
    else:
        try:
            response=api.parse(**kwargs)
            value=response.output_parsed.model_dump(mode='json') if response.output_parsed else None
            status='ok' if value else 'failed';error=None
        except Exception as exc:
            if not api.events:raise
            status='failed';value=None;error=f'{type(exc).__name__}: {exc}'
        row=dict(identity=identity,query_id=q['query_id'],repeat=1,question=q['question'],
            question_sha256=P.sha(q['question']),manifest_sha256=P.sha(manifest),request_key=key,
            status=status,error=error,plan=value,events=api.events)
        write('plans/'+q['query_id']+'.json',row)
    P.validate_parser_row(row,identity=identity,query_id=q['query_id'],repeat=1,question=q['question'],
        manifest=manifest,request_key=key,cache=api.cache)
    plans[q['query_id']]=row
    if P.ledger()['unknown_cost_records']:raise RuntimeError('추출 비용 미상: 추가 유료 실행 중단')
    print('parser',q['query_id'],row['status'],flush=True)
# LLM에는 실험용 view나 정답 주석을 보내지 않았습니다.
frozen(PATH/'parser_results.json',dict(records=[plans[q['query_id']] for q in queries],
 previous48_source_sha256=hashlib.sha256((prior/'parser_results.json').read_bytes()).hexdigest()))


parser P01 ok


parser P02 ok


parser P03 ok


parser P04 ok


In [8]:
runtime=R.Runtime(47)
vectors=runtime.embeddings([q['question'] for q in queries],allow_network=ALLOW_NETWORK)
by_q={q['query_id']:q for q in queries}
old=P.F.read_json(prior/'combined_results.json')
old_retrieval={(r['query_id'],r['profile']):r for r in old['retrievals'] if r['mode']=='S_THEN_U'}
old_answers={(r['query_id'],r['profile']):r for r in old['answers'] if r['mode']=='S_THEN_U'}
# 복사본을 매 재개 검증하고 logical index identity를 각 검색 입력에 결속합니다.
for profile in profiles:runtime.profile(profile)
retrievals=[]
for index,q in enumerate(queries):
    plan=plans[q['query_id']]['plan'];view=fact_view(q['view'])
    trusted=P.explicit_scope(q['question'],catalog)
    route=R.selected_route(q['question'],plan,view,catalog,trusted_scope=trusted,
        parser_failed=plans[q['query_id']]['status']!='ok')
    priorities=R.partial_candidates(plan,view,route)
    for profile in profiles:
        cached_A=None
        for mode in modes:
            identity=f"{q['query_id']}:{profile}:{mode}"
            destination=PATH/'retrieval'/(identity+'.json')
            binding=P.sha(dict(question=q['question'],plan=plan,view=view_audit[q['view']]['view_sha256'],
                routing=route,priorities=priorities,profile=profile,mode=mode,manifest=P.sha(manifest),index_identity=runtime.index_identity[profile]))
            if destination.exists():
                row=P.F.read_json(destination);assert row['input_binding']==binding and row['identity']==identity and row['query_id']==q['query_id'] and row['profile']==profile and row['mode']==mode
            else:
                started=time.perf_counter();reuse=None
                if q['query_id'].startswith('E') and (mode=='A' or not priorities):
                    previous=old_retrieval[q['query_id'],profile]
                    old_scope=previous['selection']['eligible_cards'] if not previous['fallback'] else plan['card_keys'] or None
                    if route['scope']==old_scope:
                        previous,verified_answer,verified_payload=R.verified_historical(q,profile,route['scope'],manifest=manifest)
                        result=copy.deepcopy(previous['result']);reuse='saved42_same_scope'
                    else:result=runtime.search(q['question'],vectors[index],profile,route['scope'])
                elif mode=='B' and not priorities:
                    result=copy.deepcopy(cached_A['result']);reuse='same_as_current_A_no_partial'
                else:
                    supplement=dict(question=q['question'],vector=vectors[index],scope=priorities) if mode=='B' and priorities else None
                    result=runtime.search(q['question'],vectors[index],profile,route['scope'],supplement=supplement,
                        baseline_result=cached_A['result'] if mode=='B' and priorities else None)
                row=dict(identity=identity,query_id=q['query_id'],profile=profile,mode=mode,status='ok',result=result,
                    route=route,partial_priorities=priorities,reuse=reuse,input_binding=binding,
                    wall_seconds=time.perf_counter()-started,source_manifest_sha256=P.sha(manifest),index_identity_sha256=P.sha(runtime.index_identity[profile]))
                write('retrieval/'+identity+'.json',row)
            if mode=='A':cached_A=row
            retrievals.append(row)
            print('retrieval',identity,row['reuse'] or 'fresh',flush=True)
write('retrieval_results.json',dict(records=retrievals,resources=runtime.resources()))
assert len(retrievals)==208 and len({r['identity'] for r in retrievals})==208
# 검색 뒤의 답변 입력만 봉인하며 정답 주석은 넣지 않습니다.
from pickcardu_rag.answering import serialize_answer_payload, _generated_answer_model
payloads={r['identity']:P.sha([dict(role='user',content=serialize_answer_payload(by_q[r['query_id']]['question'],r['result']['evidence']))]) for r in retrievals}
frozen(PATH/'answer_payload_contract.json',dict(execution_manifest_sha256=P.sha(manifest),input_sha256_by_identity=payloads,
 schema_sha256_by_identity={r['identity']:P.sha(_generated_answer_model(len(r['result']['evidence'])).model_json_schema()) for r in retrievals},
 retrieval_results_sha256=P.sha(retrievals),gold_labels_in_payload=False))
print('답변 입력208조건 봉인, 현재 API 사용량',P.ledger())


retrieval E01:card_page_section_benefit:A saved42_same_scope


retrieval E01:card_page_section_benefit:B saved42_same_scope


retrieval E01:parent_child_bundle:A saved42_same_scope


retrieval E01:parent_child_bundle:B saved42_same_scope


retrieval E02:card_page_section_benefit:A saved42_same_scope


...

retrieval P04:card_page_section_benefit:A fresh


retrieval P04:card_page_section_benefit:B fresh


retrieval P04:parent_child_bundle:A fresh


retrieval P04:parent_child_bundle:B fresh


답변 입력208조건 봉인, 현재 API 사용량 {'record_count': 90, 'known_priced_records': 90, 'unknown_cost_records': 0, 'duplicate_saved_responses': 0, 'known_cost_usd': '0.025202595', 'tokens': {'input_tokens': 300897, 'output_tokens': 28446, 'ordinary_input_tokens': 267, 'cached_input_tokens': 231561, 'cache_write_tokens': 69069, 'embedding_tokens': 183}, 'billed_cost_usd': None}


In [9]:
from pickcardu_rag.answering import OpenAIService
answers=[];by_payload={}
answer_contract=P.F.read_json(PATH/'answer_payload_contract.json')
for r in retrievals:
    identity=r['identity'];q=by_q[r['query_id']];destination=PATH/'answers'/(identity+'.json')
    payload=payloads[identity]
    historical=None;reused=None
    if q['query_id'].startswith('E') and r['reuse']=='saved42_same_scope':
        previous,historical,old_payload=R.verified_historical(q,r['profile'],r['route']['scope'],manifest=manifest)
        assert old_payload==payload and previous['result']['evidence']==r['result']['evidence']
    elif payload in by_payload:reused=by_payload[payload]
    if destination.exists():
        row=P.F.read_json(destination)
        assert row['identity']==identity and row['query_id']==r['query_id'] and row['profile']==r['profile'] and row['mode']==r['mode'] and row['retrieval_sha256']==P.sha(r) and row['input_sha256']==payload
    else:
        started=time.perf_counter();reuse=None;events=[]
        if q['query_id'].startswith('E') and r['reuse']=='saved42_same_scope':
            saved=historical
            assert saved['status']=='ok' and saved['original_answer_question']==q['question']
            answer=copy.deepcopy(saved['answer']);usage={};reuse='saved42_exact_question_evidence'
        elif reused is not None:
            previous=reused;answer=copy.deepcopy(previous['answer']);usage={};reuse=previous['identity']
        elif not r['result']['evidence']:
            answer=dict(answer_status='insufficient_evidence',answer_text='검색된 근거가 없습니다.',recommendations=[],claims=[]);usage={}
        else:
            api=R.BoundCache(47,'answer',identity,allow_network=ALLOW_NETWORK)
            try:
                parsed,usage=OpenAIService(api_key=None,client=api,llm_model='gpt-6-luna',answer_payload_bytes=64000).answer(q['question'],r['result']['evidence'])
                answer=parsed.model_dump(mode='json');error=None
            except Exception as exc:
                if not api.events:raise
                answer=None;usage={};error=f'{type(exc).__name__}: {exc}'
            events=api.events
        row=dict(identity=identity,query_id=q['query_id'],profile=r['profile'],mode=r['mode'],
            status='ok' if answer is not None else 'failed',answer=answer,usage=usage,events=events,reuse=reuse,
            retrieval_sha256=P.sha(r),input_sha256=payload,wall_seconds=time.perf_counter()-started,
            manifest_sha256=P.sha(manifest),payload_contract_sha256=P.sha(answer_contract),
            request_keys=[e['key'] for e in events],reuse_source_sha256=P.sha(historical if historical is not None else reused) if (historical is not None or reused is not None) else None)
        write('answers/'+identity+'.json',row)
    R.validate_generated_answer(row,q['question'],r['result']['evidence'],stage=47,identity=identity,
        manifest=manifest,contract=answer_contract,historical=historical,reused=reused)
    answers.append(row)
    if row['status']=='ok':by_payload[payload]=row
    if P.ledger()['unknown_cost_records']:raise RuntimeError('비용 미상: 추가 유료 실행 중단')
    write('progress.json',dict(completed=len(answers),expected=208,last_identity=identity,ledger=P.ledger()))
    print('answer',identity,row['status'],row['reuse'] or 'fresh',flush=True)
assert len(answers)==208 and len({r['identity'] for r in answers})==208
write('answer_results.json',dict(records=answers,resources=runtime.resources()))
print('완료208조건',P.ledger())


answer E01:card_page_section_benefit:A ok saved42_exact_question_evidence


answer E01:card_page_section_benefit:B ok saved42_exact_question_evidence


answer E01:parent_child_bundle:A ok saved42_exact_question_evidence


answer E01:parent_child_bundle:B ok saved42_exact_question_evidence


answer E02:card_page_section_benefit:A ok saved42_exact_question_evidence


...

answer P04:card_page_section_benefit:B ok fresh


answer P04:parent_child_bundle:A ok fresh


answer P04:parent_child_bundle:B ok fresh


완료208조건

 {'record_count': 90, 'known_priced_records': 90, 'unknown_cost_records': 0, 'duplicate_saved_responses': 0, 'known_cost_usd': '0.025202595', 'tokens': {'input_tokens': 300897, 'output_tokens': 28446, 'ordinary_input_tokens': 267, 'cached_input_tokens': 231561, 'cache_write_tokens': 69069, 'embedding_tokens': 183}, 'billed_cost_usd': None}


In [10]:
# 정답 주석은 생성이 끝난 뒤 평가 함수에만 전달합니다.
M=load('stage47_rubric','39_answer_rubric.py')
labels={q['query_id']:{**q,'plan':q['gold_plan']} for q in annotations+extra_annotations}
lookup={r['identity']:r for r in retrievals};scored=[]
for answer in answers:
    retrieval=lookup[answer['identity']];case=labels[answer['query_id']]
    result=retrieval['result'];gold=set(case['gold_cards'])
    cards=[c['card_key'] for c in result['cards']]
    first=next((i for i,k in enumerate(cards,1) if k in gold),None)
    scores=M.score(case,answer['answer'],result['evidence']) if answer['status']=='ok' else dict(
       known_positive_complete=0 if gold else None,citation_ownership=0,unsupported_percent_claims=0,explicit_scope_preserved=0)
    owners={c.chunk_id:c.card_key for c in runtime.profile(answer['profile'])[0]}
    worklist={owners[row['chunk_id']] for row in result.get('trace',{}).get('leaf',[]) if row['chunk_id'] in owners}
    scored.append(dict(identity=answer['identity'],query_id=answer['query_id'],profile=answer['profile'],mode=answer['mode'],
       subgroup=case['subgroup'],gold_anchor_hit_at_3=int(first is not None) if gold else None,
       gold_anchor_mrr=1/first if first else 0 if gold else None,gold_candidate20=sorted(gold&worklist),
       scores=scores,new_api_answer=not bool(answer['reuse']) and bool(answer['events']),
       negative_requires_source_review=case['negative_control']))
pairs=[]
for q in queries:
 for profile in profiles:
    a=next(r for r in scored if r['query_id']==q['query_id'] and r['profile']==profile and r['mode']=='A')
    b=next(r for r in scored if r['query_id']==q['query_id'] and r['profile']==profile and r['mode']=='B')
    a_row=lookup[a['identity']];b_row=lookup[b['identity']]
    pairs.append(dict(query_id=q['query_id'],profile=profile,view=q['view'],
       same_evidence=P.sha(a_row['result']['evidence'])==P.sha(b_row['result']['evidence']),
       same_cards=[c['card_key'] for c in a_row['result']['cards']]==[c['card_key'] for c in b_row['result']['cards']],
       A_complete=a['scores']['known_positive_complete'],B_complete=b['scores']['known_positive_complete'],
       candidate_anchor_loss=sorted(set(a['gold_candidate20'])-set(b['gold_candidate20'])),
       A=a,B=b))
write('evaluation_results.json',dict(records=scored,pairs=pairs,resources=runtime.resources(),
  manual_source_review_required=True,selection='pending_source_review',
  primary_limit='48개는 과거 질문·동일 입력 출력 재사용. 추가4개는 제어된DB view이며 자연 발생 오류율 아님'))
print('새 근거가 달라진 비교:',[(p['query_id'],p['profile']) for p in pairs if not p['same_evidence']])
print('자원',runtime.resources())
# 최초 유료 실행 자원은 별도 원본에 보존하며 재개 시간을 덮어쓰지 않습니다.
write('latest_replay_resources.json',dict(resources=runtime.resources(),offline=not ALLOW_NETWORK))

changed=[];allowed_appends=[]
allowance=P.F.read_json(PATH/'authorized_metric_append.json') if (PATH/'authorized_metric_append.json').exists() else None
for record in P.F.read_json(protection_path)['files']:
 source=ROOT/record['path']
 if not source.exists():changed.append(record['path']);continue
 if P.F.file_record(source)!=record:
    body=source.read_bytes()
    approved=(allowance and record['path']==allowance['path'] and P.sha(body)==allowance['current_sha256']
       and P.sha(body[:record['size']])==record['sha256'])
    if approved:allowed_appends.append(record['path'])
    else:changed.append(record['path'])
write('preservation_verification.json',dict(checked=len(P.F.read_json(protection_path)['files']),
 changed=changed,allowed_document_appends=allowed_appends,passed=not changed))
assert not changed,changed


새 근거가 달라진 비교: [('P01', 'parent_child_bundle'), ('P02', 'card_page_section_benefit'), ('P02', 'parent_child_bundle'), ('P04', 'card_page_section_benefit'), ('P04', 'parent_child_bundle')]
자원 {'wall_seconds': 76.56033579499854, 'cpu_seconds': 76.568819581, 'peak_rss_kib': 1546620, 'minimum_available_kib': None, 'new_reranker_pairs': 0, 'reranker_calls': 0, 'gpu': '미사용', 'api_ledger': {'record_count': 90, 'known_priced_records': 90, 'unknown_cost_records': 0, 'duplicate_saved_responses': 0, 'known_cost_usd': '0.025202595', 'tokens': {'input_tokens': 300897, 'output_tokens': 28446, 'ordinary_input_tokens': 267, 'cached_input_tokens': 231561, 'cache_write_tokens': 69069, 'embedding_tokens': 183}, 'billed_cost_usd': None}, 'actual_billed_cost': '미측정'}


In [11]:
# 계획에 포함된 부분일치만 강제 제한하는 위험의 오프라인 진단. 유료 행렬은 바꾸지 않습니다.
fixture=P.F.read_json(P.DATA/'43_offline_validation/routing_fixtures.json')
case=next(c for c in fixture['cases'] if c['name']=='같은카드AND')
view=[f for f in fixture['facts'] if f['document_id']!='A']
route=R.selected_route('커피 10% 할인과 넷플릭스 10% 할인을 모두 제공',case['plan'],view,fixture['catalog'])
partial=R.partial_candidates(case['plan'],view,route)
assert route['scope'] is None and 'A' not in partial
write('partial_only_diagnostic.json',dict(synthetic=True,new_api_requests=0,known_source_positive_card='A',partial_scope=partial,
 hard_partial_filter_would_exclude_positive=True,original_allowed_scope_preserved=True,
 limitation='구조화 DB view에서 A의 사실이 전부 빠진 합성 반례. 자연 오류 빈도나 P02 유료 결과로 일반화할 수 없음.'))
review=P.F.read_json(PATH/'manual_source_review.json');assert len(review['records'])==16 and all(r['citation_ownership_pass'] for r in review['records'])
for pf in profiles:
 pairs=[]
 for q in extra:
  a=next(r for r in review['records'] if r['identity']==f"{q['query_id']}:{pf}:A")
  b=next(r for r in review['records'] if r['identity']==f"{q['query_id']}:{pf}:B")
  pairs.append((int(b['label']=='PASS')-int(a['label']=='PASS')))
 print(pf,'추가4개 원문 완전성 B승/A승/동률:',pairs.count(1),pairs.count(-1),pairs.count(0))
assert P.F.read_json(PATH/'selected_policy.json')['partial_prioritization'] is False
print('47번 개발 선정: 두 프로필 모두 전체 복귀 A 유지. 운영 반영 아님.')

card_page_section_benefit 추가4개 원문 완전성 B승/A승/동률: 0 0 4
parent_child_bundle 추가4개 원문 완전성 B승/A승/동률: 0 0 4
47번 개발 선정: 두 프로필 모두 전체 복귀 A 유지. 운영 반영 아님.
